# 03 · Pruning: removing what the network does not need

**Goals of this notebook**

1. Understand why trained networks contain many unimportant parts.
2. **Edge pruning**: set individual weights (the edges of the network graph) to zero, and see why that alone does *not* make the model smaller or faster on the Arduino.
3. **Node pruning**: remove whole neurons (the nodes), so the network *really* gets smaller.
4. Learn why **fine-tuning** after pruning is essential, and compare pruning with simply training a smaller network.

In [ ]:
import sys, pathlib, copy
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import torch
import matplotlib.pyplot as plt

from aiedge.data import load_gesture_data
from aiedge.models import MLP, count_params, count_macs, count_nonzero_params
from aiedge.training import train, accuracy, set_seed
from aiedge.pruning import (magnitude_masks, sparsity_of, sparse_storage_bytes,
                            neuron_importance, remove_neurons, prune_neurons)
from aiedge import plots

data = load_gesture_data()
set_seed(0)
baseline = MLP(hidden=(64, 32))
train(baseline, data, augment=True, verbose=False)
base_acc = accuracy(baseline, data.X_test, data.y_test)
print(f"baseline: {baseline}, test accuracy {base_acc:.1%}, {count_params(baseline):,} parameters")

## 1. Why can we remove parts of a trained network?

Networks are usually trained **larger than necessary**. That makes training easier: with more weights there are more ways to reach a good solution. But once trained, many weights end up close to zero and many neurons barely contribute. Your brain does something similar: a toddler has far more connections between neurons than an adult. Unused connections are removed during childhood (*synaptic pruning*).

In a famous 2015 paper, Han et al. removed **~90% of the weights** of the image network AlexNet without losing accuracy. Later work (the *lottery ticket hypothesis*, Frankle & Carbin, 2019) even showed that big networks contain small sub-networks that could have been trained on their own.

Remember the network as a graph: **neurons are nodes, weights are edges**. We can prune either:

| | **Edge pruning** (unstructured) | **Node pruning** (structured) |
|---|---|---|
| removes | individual weights | whole neurons (with all their weights) |
| result | same matrices, many zeros | smaller matrices |
| flexibility | very high: any weight can go | lower: all or nothing per neuron |
| gain on a microcontroller | only with special sparse code | immediately: fewer MACs, less memory |

## 2. Edge pruning

### The criterion: weight magnitude
Which weights are unimportant? The simplest and very effective rule: **the ones closest to zero**. A weight of 0.001 multiplies its input by almost nothing, so setting it to exactly 0 changes the output very little.

We describe a pruned network with a **mask**: a 0/1 matrix of the same shape as each weight matrix. 1 = keep, 0 = pruned. The **sparsity** is the fraction of weights that are zero.

Let's write the pruning ourselves: find the threshold below which a fraction `sparsity` of all weights lies, and zero everything below it.

In [ ]:
def prune_smallest(model, sparsity):
    """Edge pruning: zero the `sparsity` fraction of weights with the smallest |w| (all layers together)."""
    m = copy.deepcopy(model)
    all_w = torch.cat([l.weight.detach().abs().flatten() for l in m.layers])
    threshold = torch.quantile(all_w, sparsity)
    masks = {}
    with torch.no_grad():
        for i, l in enumerate(m.layers):
            mask = (l.weight.abs() > threshold).float()
            l.weight.mul_(mask)
            masks[f"layers.{i}.weight"] = mask
    return m, masks

pruned, masks = prune_smallest(baseline, 0.5)
print(f"sparsity: {sparsity_of(pruned):.1%}   non-zero parameters: {count_nonzero_params(pruned):,} / {count_params(pruned):,}")
print(f"test accuracy after removing half of all weights: {accuracy(pruned, data.X_test, data.y_test):.1%}  (baseline {base_acc:.1%})")

Half of the weights are gone, and the accuracy hardly changed! Let's go further. We prune increasingly more, **in one shot**, without any retraining:

In [ ]:
sparsities = [0.0, 0.5, 0.7, 0.8, 0.9, 0.95, 0.98]
oneshot = []
for sp in sparsities:
    m, _ = prune_smallest(baseline, sp)
    oneshot.append(accuracy(m, data.X_test, data.y_test))
    print(f"sparsity {sp:4.0%}: test accuracy {oneshot[-1]:.1%}")

### Fine-tuning after pruning
Beyond ~80% sparsity the accuracy collapses: we are now removing weights that matter. But the remaining weights can **compensate** if we train a little more. This is **fine-tuning**.

One crucial detail: during fine-tuning the optimizer would happily give the pruned weights non-zero values again, and the pruned weights would "grow back". So after **every** optimizer step we multiply the weights by their mask again. (`train(..., masks=masks)` does this. Forgetting it is a classic bug: you think you deployed a pruned model, but it isn't sparse any more!)

In [ ]:
finetuned = []
for sp in sparsities:
    m, masks = prune_smallest(baseline, sp)
    train(m, data, masks=masks, lr=1e-3, epochs=40, patience=10, augment=True, verbose=False)
    finetuned.append(accuracy(m, data.X_test, data.y_test))
    print(f"sparsity {sp:4.0%}: one-shot {oneshot[sparsities.index(sp)]:.1%}  ->  fine-tuned {finetuned[-1]:.1%}"
          f"   (check: sparsity after fine-tuning = {sparsity_of(m):.1%})")

ax = plots.plot_curves(sparsities, {"one-shot (no fine-tuning)": oneshot, "pruned + fine-tuned": finetuned},
                       xlabel="sparsity (fraction of weights removed)", title="Edge pruning")
ax.axhline(base_acc, color=plots.MUTED, ls=":", lw=1.5); plt.show()

With fine-tuning we can remove **~90% of the weights** with little or no loss. Pruning is usually done **gradually** (prune a little, fine-tune, prune a bit more, …) for the best results, but even one step plus fine-tuning works well here.

### Where were the weights removed?
We used **one global threshold** for all layers. Let's see what fraction was removed in each layer at 90% sparsity:

In [ ]:
m90, masks90 = prune_smallest(baseline, 0.9)
for i, l in enumerate(m90.layers):
    print(f"layer {i} ({l.in_features:3d} -> {l.out_features:2d}): {sparsity_of(torch.nn.ModuleList([l])):5.1%} of its weights removed")

The first layer (300 inputs, 19 200 weights) loses the most: many of the 300 inputs are redundant (neighbouring readings in time are similar). The small last layer is almost untouched: every one of its weights matters. A global threshold finds this automatically. (`magnitude_masks(model, sparsity, scope="layer")` in `aiedge/pruning.py` prunes every layer by the same fraction instead; try it in the exercises.)

### The catch: zeros still cost memory and time
Our C code on the Arduino stores every weight matrix as a dense block of numbers and multiplies **every** weight, including the zeros. So a 90%-sparse model is exactly as big and as slow as the original!

To profit, you need a **sparse format** that only stores the non-zero weights, plus their positions. A common one is **CSR** (compressed sparse row): for every non-zero weight you store its value *and its column index*, plus one pointer per row. With int8 values and 16-bit (2-byte) indices, every surviving weight costs 3 bytes instead of 1:

In [ ]:
def dense_int8_bytes(model):
    return sum(l.weight.numel() + 4 * l.bias.numel() for l in model.layers)   # int8 weights, int32 biases

print(f"{'sparsity':>9} {'dense int8 bytes':>17} {'CSR int8 bytes':>15}")
for sp in [0.0, 0.5, 0.67, 0.8, 0.9, 0.95]:
    m, _ = prune_smallest(baseline, sp)
    print(f"{sp:>9.0%} {dense_int8_bytes(m):>17,} {sparse_storage_bytes(m, 1, 2):>15,}")

CSR only pays off above ~67% sparsity (because each kept weight costs 3 bytes instead of 1). On top of that, sparse multiplication jumps around in memory and needs extra index arithmetic per MAC, so on a small CPU it only gets *faster* at high sparsity. On big hardware such as GPUs, special sparse patterns are needed (e.g. "2 out of every 4 weights are zero").

**Edge pruning is powerful, but its gains are only real if your software and hardware can exploit sparsity.** That brings us to node pruning.

## 3. Node pruning

If an entire neuron is removed, the matrices simply become smaller. That works with any ordinary dense code, ours included.

### Which neurons are unimportant?
A neuron matters if (1) its output is often large, **and** (2) the next layer actually listens to it (large outgoing weights). We combine both into one importance score:

$$\text{importance}_j = \underbrace{\text{mean}\big(|a_j|\big)}_{\text{average output on real data}} \times \underbrace{\|W_{\text{next}}[:, j]\|}_{\text{size of its outgoing weights}}$$

In [ ]:
scores = neuron_importance(baseline, data.X_train)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.2))
for k, (ax, s) in enumerate(zip(axes, scores)):
    ax.bar(range(len(s)), s.sort(descending=True).values.numpy(), color=plots.PALETTE[0])
    ax.set_title(f"hidden layer {k + 1}: neuron importance (sorted)")
    ax.set_xlabel("neuron (sorted by importance)"); ax.set_ylabel("importance")
fig.tight_layout(); plt.show()

A handful of neurons do most of the work; many contribute little. Let's check that the score makes sense: remove the **single least** important neuron of hidden layer 1, and then (separately) the **single most** important one, and see how much the network's output changes.

### Network surgery
Removing neuron $j$ from hidden layer $l$ means:
* delete **row $j$** of the weight matrix of layer $l$ and **entry $j$** of its bias (the neuron's incoming connections);
* delete **column $j$** of the weight matrix of layer $l+1$ (its outgoing connections).

In PyTorch this is just indexing: `W_l[keep]`, `b_l[keep]`, `W_next[:, keep]`. `remove_neurons` in `aiedge/pruning.py` does it for all layers.

In [ ]:
order = scores[0].argsort()                      # least important first
all_l2 = torch.arange(32)
with torch.no_grad():
    ref = baseline(data.X_test)
    for name, drop in [("least important", order[0]), ("most important", order[-1])]:
        keep_l1 = torch.tensor([j for j in range(64) if j != drop])
        smaller = remove_neurons(baseline, [keep_l1, all_l2])
        change = (smaller(data.X_test) - ref).abs().mean()
        print(f"remove the {name:15s} neuron (#{int(drop):2d}) -> {smaller}: "
              f"mean logit change {change:.4f}, accuracy {accuracy(smaller, data.X_test, data.y_test):.1%}")

Removing the most important neuron changes the outputs several times more than removing the least important one, so the score does what it should. One neuron out of 64 barely changes the *accuracy* either way: the network is robust. But the damage adds up when we remove many neurons, as the next experiment shows.

### Pruning many neurons, with and without fine-tuning
Now we shrink both hidden layers to a range of sizes. For every size we compare three things:
1. **one-shot**: keep the most important neurons, no retraining;
2. **pruned + fine-tuned**: the same, then train a bit more;
3. **trained from scratch**: a brand-new network of the small size, trained normally.

The third one is an important *control experiment*: if it is just as good, then pruning did not help, and we could simply have trained a small network!

In [ ]:
sizes = [(64, 32), (48, 24), (32, 16), (24, 12), (16, 8), (8, 4)]
res = {"one-shot": [], "pruned + fine-tuned": [], "trained from scratch": []}
params = []
for h in sizes:
    small = prune_neurons(baseline, data.X_train, h)
    res["one-shot"].append(accuracy(small, data.X_test, data.y_test))
    train(small, data, lr=1e-3, epochs=60, patience=15, augment=True, verbose=False)
    res["pruned + fine-tuned"].append(accuracy(small, data.X_test, data.y_test))
    set_seed(0)
    scratch = MLP(hidden=h)
    train(scratch, data, augment=True, verbose=False)
    res["trained from scratch"].append(accuracy(scratch, data.X_test, data.y_test))
    params.append(count_params(small))
    print(f"{str(h):9s} {params[-1]:6,} params ({count_macs(small):6,} MACs): one-shot {res['one-shot'][-1]:.1%}, "
          f"fine-tuned {res['pruned + fine-tuned'][-1]:.1%}, from scratch {res['trained from scratch'][-1]:.1%}")

In [ ]:
ax = plots.plot_curves(params, res, xlabel="parameters (log scale)", title="Node pruning", xscale="log")
ax.axhline(base_acc, color=plots.MUTED, ls=":", lw=1.5); plt.show()

**Interpreting the results**
* **One-shot** node pruning hurts a lot: removing a neuron removes *all* its connections at once.
* **Fine-tuning** recovers most of it: down to roughly a quarter of the parameters, the pruned network is about as good as the baseline.
* **From scratch** is often just as good for a network this small! Researchers found the same for many models (Liu et al., *Rethinking the Value of Network Pruning*, 2019). Pruning helps most for big networks, when training a small one from scratch is hard. It also helps when you start from a network trained on lots of data or with a teacher (notebook 04): pruning keeps that knowledge, while a fresh network has to learn everything again.

That is a useful lesson in general: **always compare against a simple baseline** before concluding a fancy method works.

## 4. Edge vs node pruning on the Arduino

What matters for the board: accuracy versus actual bytes (int8 weights). For edge pruning we count the CSR format; for node pruning the plain dense format.

In [ ]:
edge_bytes = []
for sp in sparsities:
    m, _ = prune_smallest(baseline, sp)
    edge_bytes.append(min(sparse_storage_bytes(m, 1, 2), dense_int8_bytes(m)))   # pick the smaller format
node_bytes = [dense_int8_bytes(MLP(hidden=h)) for h in sizes]

fig, ax = plt.subplots(figsize=(6.5, 3.8))
ax.plot(np.array(edge_bytes) / 1024, finetuned, color=plots.PALETTE[0], marker="o", label="edge pruning (CSR)")
ax.plot(np.array(node_bytes) / 1024, res["pruned + fine-tuned"], color=plots.PALETTE[1], ls="--", marker="s",
        label="node pruning (dense)")
ax.set_xscale("log"); ax.set_xlabel("int8 model size [KB] (log scale)"); ax.set_ylabel("test accuracy")
ax.set_title("Accuracy vs memory after fine-tuning"); ax.legend(); plt.show()

Both reach similar sizes for similar accuracy. Node pruning has the big practical advantage that it needs **no special code**: the smaller network runs on our existing C code and is faster in proportion to its MACs. That's why notebook 05 uses **node pruning** for the deployed model.

## Summary

* Trained networks are over-parameterised; many weights and neurons can be removed.
* **Edge pruning** (smallest |w| → 0): up to ~90% sparsity with fine-tuning. But zeros only save memory/time with sparse formats and sparse kernels.
* **Node pruning** (remove whole neurons, by importance): matrices really shrink; works with any dense code.
* Always **fine-tune** after pruning, and keep the mask during fine-tuning.
* Always compare with **training the small model from scratch**.

## Exercises

1. ⭐ **Forget the mask.** Prune the baseline to 90% sparsity and fine-tune *without* `masks=`. Measure `sparsity_of(model)` after training. What happened, and why is that dangerous?
2. ⭐ **Global vs per-layer.** Use `magnitude_masks(baseline, 0.9, scope="layer")` (apply the masks by multiplying the weights, then fine-tune with `masks=`). Compare with the global version. Which is better, and why?
3. **Gradual pruning.** Shrink the baseline from (64, 32) to (16, 8) in 4 steps (48/24, 32/16, 24/12, 16/8) with a short fine-tune (20 epochs) after each step. Is it better than one-shot pruning + fine-tuning?
4. **Another importance score.** Change the score to only the L1 norm of the *incoming* weights of each neuron (`model.layers[l].weight.abs().sum(dim=1)`) and use `remove_neurons` with it. Does the choice of score matter after fine-tuning?
5. **Think.** Our input layer has 300 inputs. Node pruning cannot remove inputs. Which inputs could you remove instead, and what would that mean physically (hint: channels and time steps)?